In [ ]:
import numpy as np
import pandas as pd
from modelens import RegressionAnalyzer

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
raw_data = pd.read_csv('dataset.csv')
df = raw_data.copy()

#### 1 - EDA + ETL

In [ ]:
analyzer = RegressionAnalyzer(df,target='mpg')

In [ ]:
analyzer.info()

In [ ]:
# Check Invalid Data
df['horsepower'].unique()

In [ ]:
# ? is Invalida Data in horsepower Column
df['horsepower'] = df['horsepower'].replace({"?":np.nan}).astype(float)
df = df.dropna(subset=['horsepower'])
df['horsepower'].unique()

In [ ]:
# One Hot Encoding
df = pd.get_dummies(
    df,
    columns=["origin"],
    prefix="origin",
    drop_first=True,
    dtype=int,
)

In [ ]:
# Remove UnUsed Columns
if 'car name' in df.columns:
    df = df.drop(columns=['car name'])

In [ ]:
analyzer.reinit(df=df,target='mpg')

In [ ]:
analyzer.info()

In [ ]:
df.head()

#### 2 - Comparing Models

In [ ]:
target = 'mpg'
X = df.drop(columns=[target])
y = df[target]
features = df.columns.drop(target)
features

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    # Linear
    "Linear Regression": LinearRegression(),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ]),

    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso()),
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet()),
    ]),

    # Distance / Kernel
    "KNN": Pipeline([
        ("scaler",StandardScaler()),
        ("KNN",KNeighborsRegressor())
    ]),
    "SVR":Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR()),
    ]),

    # Tree
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
    ),

    # Bagging
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees": ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    # Boosting
    "AdaBoost": AdaBoostRegressor(
        random_state=42,
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        random_state=42,
    ),

    "XGBoost": XGBRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "LightGBM": LGBMRegressor(
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),

    "CatBoost": CatBoostRegressor(
        random_state=42,
        verbose=0,
    ),
}
# analyzer.compare_models(models=models,features=features,export_html=True) ==> ExtraTrees

#### 3 - Evaluate Data

In [ ]:
# _,suspicious_features=analyzer.correlation(features=features)

In [ ]:
# analyzer.vif()

In [ ]:
# analyzer.evaluate_single_feature_removal(features=features)

In [ ]:
selected_model = ExtraTreesRegressor()
# analyzer.evaluate_single_feature_removal(model=selected_model,features=features,export_html=True)

In [ ]:
# candidates = suspicious_features
# analyzer.evaluate_feature_removal_combinations(model=selected_model,features=features,candidates=candidates,export_html=True)

In [ ]:
# analyzer.permutation_importance(model=selected_model,export_html=True)

In [ ]:
# analyzer.check_overfitting(model=selected_model)

In [ ]:
analyzer.residual_analysis(model=selected_model,export_html=True)

In [ ]:
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt", 0.7],
}
# analyzer.tune_model(model=selected_model,param_grid=param_grid,export_html=True)

In [ ]:
analyzer.learning_curve(model=selected_model)